# Compatibility Barrier Experiment

This notebook supports the RT2 Assignment 2 project: **Breaking the Windows Compatibility Barrier: Engineering Linux Adoption through SteamOS, Bazzite, and Proton**.

## 1. Research Question

**How do Windows-specific dependencies affect the simulated probability that an application can run on Linux through a compatibility layer without native porting or full virtualisation?**

The dataset is synthetic. It is intended to operationalise the compatibility-barrier taxonomy, not to measure real-world application compatibility rates.

## 2. Hypotheses

- H1: Kernel/service dependencies reduce compatibility scores.
- H2: Dependency type matters more than application class.
- H3: Proton-like strategies are strong but domain-specific.
- H4: Managed prefixes reduce setup effort.
- H5: Virtualisation preserves compatibility but retains Windows dependence.
- H6: Distribution-level integration reduces adoption friction without directly changing API compatibility.

## 3. Synthetic Data Generation

The following cell creates a reproducible synthetic dataset with 600 rows: six application classes, five strategies and twenty trials per class-strategy combination.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

def find_project_root():
    cwd = Path.cwd()
    candidates = [cwd] + list(cwd.parents)
    for candidate in candidates:
        if (candidate / "requirements.txt").exists() or (candidate / ".git").exists():
            return candidate
    if cwd.name == "notebook":
        return cwd.parent
    return cwd

BASE_PATH = find_project_root()
RAW_DIR = BASE_PATH / "data" / "raw"
PROCESSED_DIR = BASE_PATH / "data" / "processed"
FIGURES_DIR = BASE_PATH / "figures"

for path in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

application_classes = [
    "productivity",
    "creative",
    "cad_engineering",
    "scientific",
    "enterprise_legacy",
    "games",
]

strategies = [
    "native_linux",
    "wine_like",
    "proton_like",
    "managed_prefix",
    "virtualisation",
]

profiles = {
    "productivity": {
        "api_dependency": [("low", 0.35), ("medium", 0.55), ("high", 0.10)],
        "kernel_dependency": [("none", 0.80), ("service", 0.18), ("kernel_driver", 0.02)],
        "hardware_dependency": [("none", 0.80), ("standard_device", 0.18), ("proprietary_device", 0.02)],
        "licensing_dependency": [("none", 0.30), ("offline_activation", 0.25), ("online_activation", 0.40), ("hardware_dongle", 0.05)],
    },
    "creative": {
        "api_dependency": [("low", 0.10), ("medium", 0.45), ("high", 0.45)],
        "kernel_dependency": [("none", 0.65), ("service", 0.28), ("kernel_driver", 0.07)],
        "hardware_dependency": [("none", 0.35), ("standard_device", 0.45), ("proprietary_device", 0.20)],
        "licensing_dependency": [("none", 0.15), ("offline_activation", 0.25), ("online_activation", 0.50), ("hardware_dongle", 0.10)],
    },
    "cad_engineering": {
        "api_dependency": [("low", 0.05), ("medium", 0.25), ("high", 0.70)],
        "kernel_dependency": [("none", 0.45), ("service", 0.35), ("kernel_driver", 0.20)],
        "hardware_dependency": [("none", 0.20), ("standard_device", 0.35), ("proprietary_device", 0.45)],
        "licensing_dependency": [("none", 0.05), ("offline_activation", 0.20), ("online_activation", 0.40), ("hardware_dongle", 0.35)],
    },
    "scientific": {
        "api_dependency": [("low", 0.15), ("medium", 0.50), ("high", 0.35)],
        "kernel_dependency": [("none", 0.55), ("service", 0.28), ("kernel_driver", 0.17)],
        "hardware_dependency": [("none", 0.35), ("standard_device", 0.35), ("proprietary_device", 0.30)],
        "licensing_dependency": [("none", 0.25), ("offline_activation", 0.35), ("online_activation", 0.25), ("hardware_dongle", 0.15)],
    },
    "enterprise_legacy": {
        "api_dependency": [("low", 0.05), ("medium", 0.35), ("high", 0.60)],
        "kernel_dependency": [("none", 0.45), ("service", 0.45), ("kernel_driver", 0.10)],
        "hardware_dependency": [("none", 0.55), ("standard_device", 0.35), ("proprietary_device", 0.10)],
        "licensing_dependency": [("none", 0.10), ("offline_activation", 0.20), ("online_activation", 0.55), ("hardware_dongle", 0.15)],
    },
    "games": {
        "api_dependency": [("low", 0.02), ("medium", 0.23), ("high", 0.75)],
        "kernel_dependency": [("none", 0.55), ("service", 0.25), ("kernel_driver", 0.20)],
        "hardware_dependency": [("none", 0.55), ("standard_device", 0.40), ("proprietary_device", 0.05)],
        "licensing_dependency": [("none", 0.20), ("offline_activation", 0.20), ("online_activation", 0.45), ("hardware_dongle", 0.15)],
    },
}

baseline_score = {
    "native_linux": 95,
    "virtualisation": 90,
    "proton_like": 78,
    "managed_prefix": 72,
    "wine_like": 65,
}

api_penalty = {"low": 0, "medium": 8, "high": 18}
kernel_penalty = {"none": 0, "service": 15, "kernel_driver": 35}
hardware_penalty = {"none": 0, "standard_device": 8, "proprietary_device": 22}
licensing_penalty = {"none": 0, "offline_activation": 8, "online_activation": 14, "hardware_dongle": 28}

setup_base = {
    "native_linux": 10,
    "proton_like": 35,
    "managed_prefix": 30,
    "virtualisation": 50,
    "wine_like": 65,
}

integration_base = {
    "native_linux": 95,
    "proton_like": 82,
    "managed_prefix": 76,
    "wine_like": 68,
    "virtualisation": 45,
}

overhead_base = {
    "native_linux": 2,
    "proton_like": 8,
    "managed_prefix": 10,
    "wine_like": 12,
    "virtualisation": 24,
}

def weighted_choice(options):
    values = [x[0] for x in options]
    probs = [x[1] for x in options]
    return rng.choice(values, p=probs)

def packaging_for_strategy(strategy, app_class):
    if strategy == "native_linux":
        return "app_profile"
    if strategy == "wine_like":
        return rng.choice(["raw", "managed_prefix"], p=[0.75, 0.25])
    if strategy == "proton_like":
        if app_class == "games":
            return rng.choice(["managed_prefix", "app_profile"], p=[0.35, 0.65])
        return rng.choice(["raw", "managed_prefix", "app_profile"], p=[0.20, 0.55, 0.25])
    if strategy == "managed_prefix":
        return rng.choice(["managed_prefix", "app_profile"], p=[0.65, 0.35])
    return "app_profile"

def dependency_risk(api, kernel, hardware, licensing):
    score = (
        api_penalty[api] * 1.0
        + kernel_penalty[kernel] * 1.25
        + hardware_penalty[hardware] * 0.9
        + licensing_penalty[licensing] * 1.0
    )
    return round(min(100, score), 2)

def score_profile(app_class, strategy, api, kernel, hardware, licensing, packaging):
    score = baseline_score[strategy]
    before_noise = score

    if strategy == "native_linux":
        score -= hardware_penalty[hardware] * 0.25
        score -= licensing_penalty[licensing] * 0.20
        score -= kernel_penalty[kernel] * 0.15
    elif strategy == "virtualisation":
        score -= api_penalty[api] * 0.10
        score -= kernel_penalty[kernel] * 0.25
        score -= hardware_penalty[hardware] * 0.45
        score -= licensing_penalty[licensing] * 0.25
    elif strategy == "proton_like":
        score -= api_penalty[api] * 0.45
        score -= kernel_penalty[kernel] * 0.95
        score -= hardware_penalty[hardware] * 0.80
        score -= licensing_penalty[licensing] * 0.90
        if app_class == "games":
            score += 8
        if api == "high" and app_class in {"games", "creative"}:
            score += 5
        if app_class == "enterprise_legacy":
            score -= 8
    elif strategy == "managed_prefix":
        score -= api_penalty[api] * 0.75
        score -= kernel_penalty[kernel] * 0.90
        score -= hardware_penalty[hardware] * 0.85
        score -= licensing_penalty[licensing] * 0.85
        if packaging == "app_profile":
            score += 8
        elif packaging == "managed_prefix":
            score += 5
    else:
        score -= api_penalty[api] * 0.85
        score -= kernel_penalty[kernel] * 1.00
        score -= hardware_penalty[hardware] * 0.95
        score -= licensing_penalty[licensing] * 0.95
        if packaging == "raw":
            score -= 8

    noise = rng.normal(0, 5)
    score_after = score
    score = np.clip(score + noise, 0, 100)
    return round(before_noise, 2), round(score_after, 2), round(float(score), 2)

def outcome_from_score(score):
    if score >= 75:
        return "full"
    if score >= 45:
        return "partial"
    return "failed"

def setup_effort(strategy, api, kernel, hardware, licensing, packaging):
    value = setup_base[strategy]
    value += api_penalty[api] * 0.25
    value += kernel_penalty[kernel] * 0.20
    value += hardware_penalty[hardware] * 0.25
    value += licensing_penalty[licensing] * 0.20
    if packaging == "raw":
        value += 10
    elif packaging == "app_profile":
        value -= 8
    value += rng.normal(0, 4)
    return round(float(np.clip(value, 0, 100)), 2)

def runtime_overhead(strategy, api, kernel, hardware):
    value = overhead_base[strategy]
    value += api_penalty[api] * 0.10
    value += kernel_penalty[kernel] * 0.04
    value += hardware_penalty[hardware] * 0.06
    value += rng.normal(0, 2)
    return round(float(np.clip(value, 0, 60)), 2)

def integration_score(strategy, kernel, hardware, packaging):
    value = integration_base[strategy]
    value -= kernel_penalty[kernel] * 0.15
    value -= hardware_penalty[hardware] * 0.20
    if packaging == "app_profile":
        value += 4
    elif packaging == "raw":
        value -= 6
    value += rng.normal(0, 4)
    return round(float(np.clip(value, 0, 100)), 2)

def windows_dependency_retained(strategy):
    return 1 if strategy == "virtualisation" else 0

rows = []
application_counter = 1

for app_class in application_classes:
    for strategy in strategies:
        for trial in range(1, 21):
            p = profiles[app_class]
            api = weighted_choice(p["api_dependency"])
            kernel = weighted_choice(p["kernel_dependency"])
            hardware = weighted_choice(p["hardware_dependency"])
            licensing = weighted_choice(p["licensing_dependency"])
            packaging = packaging_for_strategy(strategy, app_class)
            score_before, score_after_penalties, score = score_profile(
                app_class, strategy, api, kernel, hardware, licensing, packaging
            )
            rows.append({
                "application_id": f"APP-{application_counter:04d}",
                "trial": trial,
                "application_class": app_class,
                "strategy": strategy,
                "api_dependency": api,
                "kernel_dependency": kernel,
                "hardware_dependency": hardware,
                "licensing_dependency": licensing,
                "packaging_support": packaging,
                "compatibility_outcome": outcome_from_score(score),
                "compatibility_score": score,
                "setup_effort": setup_effort(strategy, api, kernel, hardware, licensing, packaging),
                "runtime_overhead": runtime_overhead(strategy, api, kernel, hardware),
                "integration_score": integration_score(strategy, kernel, hardware, packaging),
                "windows_dependency_retained": windows_dependency_retained(strategy),
                "random_seed": RANDOM_SEED,
                "score_before_noise": score_before,
                "score_after_penalties": score_after_penalties,
                "dependency_risk_index": dependency_risk(api, kernel, hardware, licensing),
                "notes": "synthetic row; not a real application benchmark",
            })
            application_counter += 1

df = pd.DataFrame(rows)
raw_path = RAW_DIR / "synthetic_trials.csv"
df.to_csv(raw_path, index=False)

print(f"Saved raw dataset to: {raw_path}")
print(f"Rows: {len(df)}")
df.head()

## 4. Dataset Validation

In [ ]:
import pandas as pd

raw_path = RAW_DIR / "synthetic_trials.csv"
df = pd.read_csv(raw_path)

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

df.head()

## 5. Aggregated Results

In [ ]:
strategy_summary = (
    df.groupby("strategy")
    .agg(
        trials=("application_id", "count"),
        mean_compatibility_score=("compatibility_score", "mean"),
        std_compatibility_score=("compatibility_score", "std"),
        median_setup_effort=("setup_effort", "median"),
        mean_runtime_overhead=("runtime_overhead", "mean"),
        mean_integration_score=("integration_score", "mean"),
        retained_windows_dependency_rate=("windows_dependency_retained", "mean"),
    )
    .reset_index()
)

success_rate = (
    df.assign(full_compatibility=(df["compatibility_outcome"] == "full").astype(int))
    .groupby("strategy")["full_compatibility"]
    .mean()
    .reset_index(name="full_compatibility_rate")
)

strategy_summary = strategy_summary.merge(success_rate, on="strategy").round(3)

processed_path = PROCESSED_DIR / "aggregated_results.csv"
strategy_summary.to_csv(processed_path, index=False)

print(f"Saved processed results to: {processed_path}")
strategy_summary

## 6. Outcome Distribution

In [ ]:
outcome_table = (
    df.pivot_table(
        index="strategy",
        columns="compatibility_outcome",
        values="application_id",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(strategies)
)

outcome_rate_table = outcome_table.div(outcome_table.sum(axis=1), axis=0).round(3)

print("Outcome counts:")
display(outcome_table)

print("Outcome rates:")
display(outcome_rate_table)

## 7. Figures

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: full compatibility rate by strategy.
plot_df = success_rate.sort_values("full_compatibility_rate", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df["strategy"], plot_df["full_compatibility_rate"])
plt.ylabel("Full compatibility rate")
plt.xlabel("Strategy")
plt.title("Full Compatibility Rate by Strategy")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "compatibility_success_rate.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: strategy comparison.
comparison = strategy_summary.set_index("strategy")[
    ["mean_compatibility_score", "median_setup_effort", "mean_runtime_overhead", "mean_integration_score"]
].copy()
comparison["mean_runtime_overhead"] = comparison["mean_runtime_overhead"] * 2

plt.figure(figsize=(10, 5))
comparison.plot(kind="bar")
plt.ylabel("Score / scaled value")
plt.xlabel("Strategy")
plt.title("Strategy Comparison")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "strategy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 3: heatmap for kernel dependency.
risk_table = (
    df.groupby(["strategy", "kernel_dependency"])["compatibility_score"]
    .mean()
    .unstack()
    .reindex(strategies)
)

plt.figure(figsize=(8, 5))
plt.imshow(risk_table.fillna(0).values, aspect="auto")
plt.xticks(range(len(risk_table.columns)), risk_table.columns, rotation=30, ha="right")
plt.yticks(range(len(risk_table.index)), risk_table.index)
plt.colorbar(label="Mean compatibility score")
plt.title("Dependency Risk Heatmap: Kernel Dependency")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dependency_risk_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 4: setup effort boxplot.
plt.figure(figsize=(8, 5))
data_for_box = [df[df["strategy"] == s]["setup_effort"] for s in strategies]
plt.boxplot(data_for_box, tick_labels=strategies)
plt.ylabel("Setup effort")
plt.xlabel("Strategy")
plt.title("Setup Effort Distribution by Strategy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "setup_effort_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 5: integration score by strategy.
integration = df.groupby("strategy")["integration_score"].mean().reindex(strategies)

plt.figure(figsize=(8, 5))
plt.bar(integration.index, integration.values)
plt.ylabel("Mean integration score")
plt.xlabel("Strategy")
plt.title("Linux Integration Score by Strategy")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "integration_score_by_strategy.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Figures saved to: {FIGURES_DIR}")

## 8. Exploratory Statistical Tests

These tests are included only as a structured comparison of the synthetic model. They must not be interpreted as empirical proof about the real Windows software ecosystem.

In [ ]:
from scipy.stats import chi2_contingency, kruskal

# These tests are exploratory because the dataset is synthetic.
contingency = pd.crosstab(df["strategy"], df["compatibility_outcome"])
chi2, p_value, dof, expected = chi2_contingency(contingency)

print("Chi-square test: strategy vs compatibility_outcome")
print("chi2:", round(chi2, 3))
print("p-value:", round(p_value, 5))
print("degrees of freedom:", dof)

samples = [df[df["strategy"] == s]["compatibility_score"] for s in strategies]
kw_stat, kw_p = kruskal(*samples)

print("\nKruskal-Wallis test: compatibility_score across strategies")
print("statistic:", round(kw_stat, 3))
print("p-value:", round(kw_p, 5))

## 9. Limitations

This experiment uses synthetic data, manually defined weights and simplified dependency categories. It should be interpreted as a methodological illustration supporting the state-of-the-art discussion, not as a real benchmark of Proton, Wine, Bazzite or Linux application compatibility.